# Day 02: Deep Autoencoder (Hinton & Salakhutdinov, Science 2006)
### Sổ Tay Thực Hành From Scratch: `autoencoder_scratch.ipynb`

> **Bài báo gốc:** *Reducing the Dimensionality of Data with Neural Networks*  
> **Tác giả:** Geoffrey E. Hinton & Ruslan R. Salakhutdinov (Đại học Toronto)  
> **Năm công bố:** 2006 — Tạp chí danh giá *Science*  
> **Tài liệu tham khảo:** [`resources/Day 2 Autoencoder.pdf`](./resources/Day%202%20Autoencoder.pdf)

---

## 🎯 Mục Tiêu Cốt Lõi Của Notebook
1. **Giải thích chi tiết cho người mới bắt đầu:** Hiểu rõ tại sao mạng nơ-ron sâu trước năm 2006 không thể huấn luyện được (Vanishing Gradient & cực tiểu địa phương tồi), và cách giải thuật **Greedy Layer-wise Pre-training** bằng Cỗ máy Boltzmann Giới hạn (RBM) đã giải cứu Deep Learning.
2. **Tự lập trình từ con số 0 (From Scratch thuần NumPy):** Xây dựng toàn bộ các tầng Dense với Backpropagation thủ công, mô hình RBM với Contrastive Divergence (CD-1), và quy trình 3 bước: Pre-train $\to$ Unroll $\to$ Fine-tune.
3. **Thực nghiệm đối chuẩn giá trị học thuật:** Tái hiện và kiểm chứng định lượng sự tiến hóa:
   - **Linear PCA (1901):** Giảm chiều tuyến tính phẳng, làm đè bẹp các đa tạp cong.
   - **Shallow Autoencoder (1986):** Mạng nông 1 tầng ẩn, bị giới hạn xấp xỉ tương tự PCA.
   - **Vanilla Deep Autoencoder (Trước 2006):** Mạng sâu không pre-train, bị kẹt cực tiểu tồi.
   - **Hinton Deep Autoencoder (2006):** Pre-training từng tầng giúp giảm sâu sai số tái tạo (MSE).

## 🏛️ Bối Cảnh Lịch Sử: Trước và Sau Khi Bài Báo Ra Đời

| Tiêu Chí So Sánh | Linear PCA (Pearson 1901) | Shallow AE (Rumelhart 1986) | Deep AE Không Pre-train | Hinton Deep AE (Science 2006) |
|---|---|---|---|---|
| **Bản chất toán học** | Tuyến tính phẳng (SVD/Eigen) | Phi tuyến nông (1 tầng ẩn) | Phi tuyến sâu (Nhiều tầng Sigmoid) | **Phi tuyến sâu (Deep Manifold)** |
| **Đa tạp cong (Curved Manifolds)** | Thất bại (đè bẹp các nhánh) | Kém, xấp xỉ không gian con PCA | Không hội tụ được đến nghiệm tốt | **Học mượt mà các cấu trúc cong** |
| **Khởi tạo trọng số** | Tính toán giải tích ma trận | Ngẫu nhiên | Ngẫu nhiên (kẹt cực tiểu địa phương) | **RBM Greedy Layer-wise Pre-training** |
| **Hiện tượng Gradient** | Không áp dụng | Đạo hàm truyền tốt (1 tầng) | **Triệt tiêu gradient (Vanishing)** | **Tránh được hoàn toàn nhờ Pre-training** |
| **Chất lượng tái tạo** | Tương đối mờ / Biến dạng | Tương tự PCA | Tái tạo trung bình xám | **Sắc nét, bảo toàn cấu trúc dữ liệu** |

In [ ]:
import time
import numpy as np

print("[✓] Đã khởi tạo môi trường tính toán số học thuần NumPy!")

## PHẦN 1: CÁC KHỐI TOÁN HỌC & TẦNG TRUYỀN THẲNG CÓ BACKPROP THỦ CÔNG

Mỗi tầng liên kết đầy đủ (Dense Layer) thực hiện phép biến đổi:
$$z = X \cdot W + b, \quad a = \text{activation}(z)$$

Trong quá trình lan truyền ngược (Backward pass), ta tính đạo hàm theo quy tắc chuỗi (Chain Rule):
$$\frac{\partial \mathcal{L}}{\partial z} = \frac{\partial \mathcal{L}}{\partial a} \odot \sigma'(z)$$
$$\frac{\partial \mathcal{L}}{\partial W} = X^T \cdot \frac{\partial \mathcal{L}}{\partial z}, \quad \frac{\partial \mathcal{L}}{\partial b} = \sum_{\text{batch}} \frac{\partial \mathcal{L}}{\partial z}, \quad \frac{\partial \mathcal{L}}{\partial X} = \frac{\partial \mathcal{L}}{\partial z} \cdot W^T$$

In [ ]:
def sigmoid(x):
    x_clipped = np.clip(x, -500, 500)
    return 1.0 / (1.0 + np.exp(-x_clipped))

def sigmoid_derivative(output):
    return output * (1.0 - output)

def linear(x):
    return x

def linear_derivative(output):
    return np.ones_like(output)

class MSELoss:
    @staticmethod
    def forward(y_pred, y_true):
        return 0.5 * np.mean(np.sum((y_pred - y_true) ** 2, axis=1))

    @staticmethod
    def gradient(y_pred, y_true):
        return (y_pred - y_true) / y_pred.shape[0]

class DenseLayer:
    def __init__(self, in_features, out_features, activation="sigmoid"):
        self.in_features = in_features
        self.out_features = out_features
        scale = np.sqrt(2.0 / (in_features + out_features))
        self.W = np.random.randn(in_features, out_features) * scale
        self.b = np.zeros((1, out_features))
        self.input_cache = None
        self.output_cache = None
        self.v_W = np.zeros_like(self.W)
        self.v_b = np.zeros_like(self.b)
        if activation == "sigmoid":
            self.act_fn, self.act_deriv = sigmoid, sigmoid_derivative
        else:
            self.act_fn, self.act_deriv = linear, linear_derivative

    def forward(self, X):
        self.input_cache = X
        z = np.dot(X, self.W) + self.b
        self.output_cache = self.act_fn(z)
        return self.output_cache

    def backward(self, grad_output):
        grad_z = grad_output * self.act_deriv(self.output_cache)
        self.grad_W = np.dot(self.input_cache.T, grad_z)
        self.grad_b = np.sum(grad_z, axis=0, keepdims=True)
        return np.dot(grad_z, self.W.T)

    def update_params(self, lr=0.01, momentum=0.9, weight_decay=1e-4):
        self.v_W = momentum * self.v_W - lr * (self.grad_W + weight_decay * self.W)
        self.v_b = momentum * self.v_b - lr * self.grad_b
        self.W += self.v_W
        self.b += self.v_b

print("[✓] Đã khởi tạo các hàm toán học và lớp DenseLayer với Backpropagation thủ công!")

## PHẦN 2: CỖ MÁY BOLTZMANN GIỚI HẠN (RBM) & CONTRASTIVE DIVERGENCE (CD-1)

RBM là mô hình đồ thị xác suất 2 tầng không có liên kết nội bộ trong cùng một tầng. Năng lượng của cấu hình $(v, h)$ được định nghĩa:
$$E(v, h) = - v^T W h - b_v^T v - b_h^T h$$

Thuật toán **Contrastive Divergence (CD-1)** của Hinton cập nhật trọng số dựa trên chênh lệch giữa kỳ vọng dữ liệu thật và kỳ vọng mô hình tự tưởng tượng (1 bước lấy mẫu Gibbs):
$$\Delta W = \epsilon \left( \langle v_0 h_0^T \rangle_{\text{data}} - \langle v_1 h_1^T \rangle_{\text{recon}} \right)$$

In [ ]:
class RBM:
    """Cỗ máy Boltzmann Giới hạn với giải thuật Contrastive Divergence (CD-1)"""
    def __init__(self, num_visible, num_hidden, is_top_linear=False):
        self.num_visible = num_visible
        self.num_hidden = num_hidden
        self.is_top_linear = is_top_linear
        self.W = np.random.randn(num_visible, num_hidden) * 0.01
        self.b_v = np.zeros((1, num_visible))
        self.b_h = np.zeros((1, num_hidden))
        self.v_W = np.zeros_like(self.W)
        self.v_b_v = np.zeros_like(self.b_v)
        self.v_b_h = np.zeros_like(self.b_h)

    def sample_hidden(self, v):
        activation = np.dot(v, self.W) + self.b_h
        if self.is_top_linear:
            h_prob = activation
            h_sample = h_prob + np.random.randn(*h_prob.shape)
        else:
            h_prob = sigmoid(activation)
            h_sample = (np.random.rand(*h_prob.shape) < h_prob).astype(np.float64)
        return h_prob, h_sample

    def sample_visible(self, h):
        activation = np.dot(h, self.W.T) + self.b_v
        v_prob = sigmoid(activation)
        v_sample = (np.random.rand(*v_prob.shape) < v_prob).astype(np.float64)
        return v_prob, v_sample

    def contrastive_divergence_step(self, v0, lr=0.1, momentum=0.5, weight_decay=2e-4):
        batch_size = v0.shape[0]
        h0_prob, h0_sample = self.sample_hidden(v0)
        positive_grad = np.dot(v0.T, h0_prob)

        v1_prob, v1_sample = self.sample_visible(h0_sample)
        h1_prob, h1_sample = self.sample_hidden(v1_prob)
        negative_grad = np.dot(v1_prob.T, h1_prob)

        dW = (positive_grad - negative_grad) / batch_size
        db_v = np.mean(v0 - v1_prob, axis=0, keepdims=True)
        db_h = np.mean(h0_prob - h1_prob, axis=0, keepdims=True)

        self.v_W = momentum * self.v_W + lr * (dW - weight_decay * self.W)
        self.v_b_v = momentum * self.v_b_v + lr * db_v
        self.v_b_h = momentum * self.v_b_h + lr * db_h

        self.W += self.v_W
        self.b_v += self.v_b_v
        self.b_h += self.v_b_h
        return np.mean((v0 - v1_prob) ** 2)

print("[✓] Đã khởi tạo thành công lớp RBM!")

## PHẦN 3: DEEP AUTOENCODER HOÀN CHỈNH (QUY TRÌNH 3 BƯỚC CỦA HINTON)

In [ ]:
class DeepAutoencoderFromScratch:
    def __init__(self, layer_dims):
        self.layer_dims = layer_dims
        self.num_stages = len(layer_dims) - 1
        self.rbms = [
            RBM(layer_dims[i], layer_dims[i+1], is_top_linear=(i == self.num_stages - 1))
            for i in range(self.num_stages)
        ]
        self.encoder_layers = [
            DenseLayer(layer_dims[i], layer_dims[i+1], activation="linear" if i == self.num_stages - 1 else "sigmoid")
            for i in range(self.num_stages)
        ]
        reversed_dims = list(reversed(layer_dims))
        self.decoder_layers = [
            DenseLayer(reversed_dims[i], reversed_dims[i+1], activation="sigmoid")
            for i in range(self.num_stages)
        ]

    def pretrain_rbms(self, X, epochs=12, lr=0.08, batch_size=32, verbose=True):
        current_input = X.copy()
        N = X.shape[0]
        for stage, rbm in enumerate(self.rbms):
            if verbose:
                print(f"   * Pre-train RBM {stage+1}/{self.num_stages}: [{rbm.num_visible} -> {rbm.num_hidden}]")
            for epoch in range(1, epochs + 1):
                indices = np.random.permutation(N)
                num_batches = int(np.ceil(N / batch_size))
                momentum = 0.5 if epoch <= 5 else 0.9
                for b in range(num_batches):
                    batch_idx = indices[b * batch_size : min((b + 1) * batch_size, N)]
                    rbm.contrastive_divergence_step(current_input[batch_idx], lr=lr, momentum=momentum)
            h_prob, _ = rbm.sample_hidden(current_input)
            current_input = h_prob

    def unroll_rbms(self):
        for i, rbm in enumerate(self.rbms):
            self.encoder_layers[i].W = rbm.W.copy()
            self.encoder_layers[i].b = rbm.b_h.copy()
        for i in range(self.num_stages):
            source_rbm = self.rbms[self.num_stages - 1 - i]
            self.decoder_layers[i].W = source_rbm.W.T.copy()
            self.decoder_layers[i].b = source_rbm.b_v.copy()

    def encode(self, X):
        out = X
        for l in self.encoder_layers:
            out = l.forward(out)
        return out

    def decode(self, Z):
        out = Z
        for l in self.decoder_layers:
            out = l.forward(out)
        return out

    def forward(self, X):
        z = self.encode(X)
        return self.decode(z), z

    def finetune_backprop(self, X, epochs=30, lr=0.03, batch_size=32, verbose=True):
        N = X.shape[0]
        for epoch in range(1, epochs + 1):
            indices = np.random.permutation(N)
            num_batches = int(np.ceil(N / batch_size))
            for b in range(num_batches):
                batch_idx = indices[b * batch_size : min((b + 1) * batch_size, N)]
                X_batch = X[batch_idx]
                x_hat, _ = self.forward(X_batch)
                grad = MSELoss.gradient(x_hat, X_batch)
                for l in reversed(self.decoder_layers):
                    grad = l.backward(grad)
                for l in reversed(self.encoder_layers):
                    grad = l.backward(grad)
                for l in self.decoder_layers:
                    l.update_params(lr=lr)
                for l in self.encoder_layers:
                    l.update_params(lr=lr)

print("[✓] Đã khởi tạo thành công lớp DeepAutoencoderFromScratch!")

## PHẦN 4: THỰC NGHIỆM ĐỐI CHUẨN GIÁ TRỊ HỌC THUẬT (ACADEMIC VALUE BENCHMARK)

Chúng ta tiến hành tạo ra một bài toán đa tạp phi tuyến 64 chiều (tương tự không gian ảnh $8 \times 8$) được sinh ra từ 2 biến tiềm ẩn $z = [t_1, t_2]$.

Ta đối chuẩn 4 cột mốc công nghệ:
1. **Linear PCA (1901)**
2. **Shallow Autoencoder (1986)**
3. **Vanilla Deep Autoencoder (Trước 2006)**: Khởi tạo ngẫu nhiên, không có Pre-training.
4. **Hinton Deep Autoencoder (Science 2006)**: Với Greedy RBM Pre-training.

In [ ]:
# 1. Sinh tập dữ liệu đa tạp phi tuyến
np.random.seed(42)
num_samples = 700
t1 = np.random.uniform(-np.pi, np.pi, size=(num_samples, 1))
t2 = np.random.uniform(-np.pi, np.pi, size=(num_samples, 1))
X = np.zeros((num_samples, 64))
for i in range(64):
    freq1, freq2 = (i % 8 + 1) * 0.5, (i // 8 + 1) * 0.5
    X[:, i] = np.sin(freq1 * t1[:, 0]) * np.cos(freq2 * t2[:, 0]) + 0.4 * np.cos(freq1 * t2[:, 0])

X = 0.05 + 0.9 * ((X - X.min()) / (X.max() - X.min()))
split = 500
X_train, X_test = X[:split], X[split:]
print(f"[*] Dữ liệu Train = {X_train.shape}, Test = {X_test.shape}")

In [ ]:
# --------------------------------------------------------------------------
# HUẤN LUYỆN VÀ ĐỐI CHUẨN 4 CỘT MỐC CÔNG NGHỆ
# --------------------------------------------------------------------------

# 1. Linear PCA (Pearson 1901)
t0 = time.time()
mean = np.mean(X_train, axis=0, keepdims=True)
_, _, Vt = np.linalg.svd(X_train - mean, full_matrices=False)
V = Vt[:2].T
pca_recon = np.dot(np.dot(X_test - mean, V), V.T) + mean
pca_mse = np.mean((X_test - pca_recon) ** 2)
pca_time = time.time() - t0

# 2. Shallow Autoencoder 64 -> 2 -> 64 (Rumelhart 1986)
t0 = time.time()
shallow_enc = DenseLayer(64, 2, activation="linear")
shallow_dec = DenseLayer(2, 64, activation="sigmoid")
for _ in range(30):
    sh_hat = shallow_dec.forward(shallow_enc.forward(X_train))
    grad = shallow_enc.backward(shallow_dec.backward(MSELoss.gradient(sh_hat, X_train)))
    shallow_dec.update_params(lr=0.04)
    shallow_enc.update_params(lr=0.04)
shallow_recon = shallow_dec.forward(shallow_enc.forward(X_test))
shallow_mse = np.mean((X_test - shallow_recon) ** 2)
shallow_time = time.time() - t0

# 3. Vanilla Deep Autoencoder 64->32->16->2 KHÔNG CÓ Pre-training (Trước 2006)
t0 = time.time()
np.random.seed(42)
vanilla_ae = DeepAutoencoderFromScratch([64, 32, 16, 2])
vanilla_ae.finetune_backprop(X_train, epochs=30, lr=0.03, verbose=False)
vanilla_recon, _ = vanilla_ae.forward(X_test)
vanilla_mse = np.mean((X_test - vanilla_recon) ** 2)
vanilla_time = time.time() - t0

# 4. Hinton Deep Autoencoder CÓ Greedy RBM Pre-training (Science 2006)
t0 = time.time()
np.random.seed(42)
hinton_ae = DeepAutoencoderFromScratch([64, 32, 16, 2])
hinton_ae.pretrain_rbms(X_train, epochs=12, lr=0.08, verbose=False)
hinton_ae.unroll_rbms()
hinton_ae.finetune_backprop(X_train, epochs=30, lr=0.03, verbose=False)
hinton_recon, _ = hinton_ae.forward(X_test)
hinton_mse = np.mean((X_test - hinton_recon) ** 2)
hinton_time = time.time() - t0

# BẢNG ĐỐI CHUẨN ĐỊNH LƯỢNG
print("=" * 90)
print(f"{'Mô Hình / Giải Thuật':<28} | {'Bottleneck':<10} | {'Test MSE':<12} | {'Cải thiện vs PCA':<18} | {'Thời gian':<10}")
print("=" * 90)
print(f"{'1. Linear PCA (1901)':<28} | {'2 chiều':<10} | {pca_mse:10.6f}   | {'Baseline (0.0%)':<18} | {pca_time:7.3f}s")
print(f"{'2. Shallow AE (1986)':<28} | {'2 chiều':<10} | {shallow_mse:10.6f}   | {((pca_mse - shallow_mse)/pca_mse * 100):+15.2f}% | {shallow_time:7.3f}s")
print(f"{'3. Vanilla Deep AE (Pre-2006)':<28} | {'2 chiều':<10} | {vanilla_mse:10.6f}   | {((pca_mse - vanilla_mse)/pca_mse * 100):+15.2f}% | {vanilla_time:7.3f}s")
print(f"{'4. Hinton Deep AE (2006)':<28} | {'2 chiều':<10} | {hinton_mse:10.6f}   | {((pca_mse - hinton_mse)/pca_mse * 100):+15.2f}% | {hinton_time:7.3f}s")
print("=" * 90)

## PHẦN 5: TRỰC QUAN HÓA SO SÁNH HÌNH ẢNH TÁI TẠO (ASCII)

Hãy quan sát cách Deep Autoencoder của Hinton tái tạo lại mẫu ảnh $8 \times 8$ sắc nét hơn rất nhiều so với PCA:

In [ ]:
chars = " .:-=+*#%@"
def to_ascii(arr, size=8):
    return ["".join([chars[int(np.clip(arr[r * size + c] * (len(chars) - 1), 0, len(chars) - 1))] * 2 for c in range(size)]) for r in range(size)]

sample_idx = 5
o_grid = to_ascii(X_test[sample_idx])
p_grid = to_ascii(pca_recon[sample_idx])
h_grid = to_ascii(hinton_recon[sample_idx])

print("\n   [Ảnh Gốc 8x8]               [Tái Tạo Bằng PCA]          [Tái Tạo Bằng Deep AE Hinton]")
for o, p, h in zip(o_grid, p_grid, h_grid):
    print(f"   {o}       {p}       {h}")

## 🎓 KẾT LUẬN & ĐÚC KẾT HỌC THUẬT DÀNH CHO NGƯỜI MỚI BẮT ĐẦU

1. **Sự thất bại của giải pháp tuyến tính:**
   - PCA chiếu vuông góc dữ liệu lên một mặt phẳng, làm đè bẹp các vòng xoắn của đa tạp phi tuyến, dẫn tới sai số tái tạo lớn.
2. **Tác dụng thần kỳ của Pre-training từng tầng (Hinton 2006):**
   - Khởi tạo ngẫu nhiên khiến mạng sâu bị tắc nghẽn bởi hiện tượng triệt tiêu gradient (Vanishing Gradient qua hàm Sigmoid) và sa lầy vào cực tiểu địa phương tồi.
   - Greedy Layer-wise Pre-training với RBM đóng vai trò như một cơ chế **khởi tạo trọng số thông minh**, hướng mạng sâu vào vùng hấp dẫn tối ưu, giúp Backpropagation phát huy tối đa sức mạnh!